# 3.7 — Marker Retargeting

`myosuite.core.retargeting.retarget` turns labelled marker trajectories into a joint-space `MotionClip`:

```python
clip = retarget(model, "trial.trc")  # a TRC or C3D file, read in its own units
clip = retarget(model, "trial.trc", sites={"RHJC": "hip_r"})  # marker -> site
clip = retarget(model, {"wrist": xyz}, fps=100)  # arrays (T, 3) in metres
```

Each frame is a `solve_ik` fit seeded with the previous frame's pose. The clip holds `qpos`, `qvel`, the fitted site positions and, in `clip.metadata`, the per-frame marker error and (for models with a free root) the sole height.

Here a two-link arm moves with joint angles known in closed form, so the result can be checked against the truth. The helper is `files/3.7/marker_retargeting.py`.

**Prerequisites:** `myosuite` installed · notebook 3.2 helps

In [ ]:
import os; _nb = globals().get("__vsc_ipynb_file__"); _nb and os.chdir(os.path.dirname(_nb))  # VS Code: work relative to this notebook

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from myosuite.core.retargeting import retarget
from myosuite.utils.video_io import show_video, write_video

helper_dir = next(p for p in (Path("files/3.7"), Path("tutorials/files/3.7")) if p.exists())
sys.path.insert(0, str(helper_dir.resolve()))
import marker_retargeting as demo  # noqa: E402  (two-link arm, closed-form angles and markers)

model = demo.build_model()
t, angles = demo.closed_form_angles()
markers = demo.markers_from_angles(angles)
print({name: xyz.shape for name, xyz in markers.items()}, f"at {demo.FPS:g} Hz")

## Retarget

The markers are named like the model's sites (`elbow`, `wrist`), so no mapping is needed. Arrays carry no frame rate: pass `fps`.

In [ ]:
clip = retarget(model, markers, fps=demo.FPS)
print(f"max joint-angle error: {np.abs(clip.qpos - angles).max():.1e} rad")
print(f"max marker error: {np.nanmax(clip.metadata['marker_error']):.1e} m")
print("qpos", clip.qpos.shape, "qvel", clip.qvel.shape, "sites", clip.site_names, f"{clip.frequency_hz:g} Hz")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharex=True)
for i, name in enumerate(("shoulder", "elbow")):
    axes[i].plot(t, angles[:, i], "k", lw=4, alpha=0.3, label="closed form")
    axes[i].plot(t, clip.qpos[:, i], "C0--", label="retarget")
    axes[i].set(title=name, xlabel="time (s)", ylabel="angle (rad)")
axes[0].legend()
plt.tight_layout()
plt.show()

## Wrong units

Exports in millimetres are a common mistake. Files are converted from the units in their header, but arrays must already be in metres. With millimetres the markers are a thousand times too far away, which the marker error shows at once.

In [ ]:
in_mm = {name: 1000 * xyz for name, xyz in markers.items()}
wrong = retarget(model, in_mm, fps=demo.FPS)
print(f"median marker error, millimetre input: {np.median(wrong.metadata['marker_error']):.0f} m")
print(f"median marker error, metre input:      {np.median(clip.metadata['marker_error']):.1e} m")

## A missing marker

NaN marks a missing sample. Here the wrist marker drops out for half a second. Those frames are fitted to the elbow marker alone: the shoulder angle still follows, but nothing observes the elbow joint, so it holds its last value until the wrist reappears. The gap shows as NaN in `marker_error`.

In [ ]:
gappy = {name: xyz.copy() for name, xyz in markers.items()}
gappy["wrist"][60:110] = np.nan
gap = retarget(model, gappy, fps=demo.FPS)
print("frames without the wrist marker:", int(np.isnan(gap.metadata["marker_error"][:, 1]).sum()))
print(f"max shoulder error: {np.abs(gap.qpos[:, 0] - angles[:, 0]).max():.1e} rad")

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(t, angles[:, 1], "k", lw=4, alpha=0.3, label="closed form")
ax.plot(t, gap.qpos[:, 1], "C1--", label="retarget, wrist missing")
ax.axvspan(t[60], t[109], color="C1", alpha=0.1)
ax.set(xlabel="time (s)", ylabel="elbow angle (rad)")
ax.legend()
plt.tight_layout()
plt.show()

## Lab marker names

When marker names differ from site names, map them with `sites`.

In [ ]:
lab = {"R_ELB": markers["elbow"], "R_WRI": markers["wrist"]}
named = retarget(model, lab, fps=demo.FPS, sites={"R_ELB": "elbow", "R_WRI": "wrist"})
print(named.metadata["marker_sites"], f"max angle error {np.abs(named.qpos - angles).max():.1e} rad")

## Replay

`clip.qpos` is a joint trajectory of this model: set it frame by frame and run forward kinematics.

In [ ]:
replayed = demo.replay(model, clip.qpos)
print(f"max replayed wrist error: {np.abs(replayed['wrist'] - markers['wrist']).max():.1e} m")
frames = demo.render(model, clip.qpos)
Path("videos").mkdir(exist_ok=True)
write_video("videos/marker_retargeting.mp4", frames, fps=demo.FPS / 2, outputdict={"-pix_fmt": "yuv420p"})
show_video("videos/marker_retargeting.mp4")

## Files and full-body models

* Pass a path instead of arrays: `retarget(model, "trial.trc")` reads an OpenSim TRC in its header units and frame rate. OpenSim is Y-up while MuJoCo models are Z-up, so add `rotation=[[1, 0, 0], [0, 0, -1], [0, 1, 0]]`. C3D files need `pip install "myosuite[capture]"`.
* For a model with a free root, the whole clip is shifted vertically so that its lowest sole point (the collision geometry of the feet) touches `z = 0`. `ground_sites=[...]` places chosen sites instead, and `ground_sites=[]` keeps the fitted height. `clip.metadata["sole_height"]` records the height per frame.
* For a scaled subject model, build the model first and retarget against it.

Set `MUJOCO_GL=egl` (or `osmesa`) on headless machines.